# SCF Phase 2 shard 32

Sweeps: alignment. Jobs: 6. Projected: 4.6 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.0, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6215583ffd95\", \"mode\": \"alignment\", \"sweep\": \"alignment\", \"reps\": 800, \"raw_path\": \"data/sim/alignment/raw/6215583ffd95.parquet\", \"means_path\": \"data/sim/alignment/means/6215583ffd95.npz\", \"_rank\": 5, \"_sweep\": \"alignment\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.832595714594046, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.0, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"9eed6407785c\", \"mode\": \"alignment\", \"sweep\": \"alignment\", \"reps\": 800, \"raw_path\": \"data/sim/alignment/raw/9eed6407785c.parquet\", \"means_path\": \"data/sim/alignment/means/9eed6407785c.npz\", \"_rank\": 5, \"_sweep\": \"alignment\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 2.0943951023931953, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.0, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"d02b148345c3\", \"mode\": \"alignment\", \"sweep\": \"alignment\", \"reps\": 800, \"raw_path\": \"data/sim/alignment/raw/d02b148345c3.parquet\", \"means_path\": \"data/sim/alignment/means/d02b148345c3.npz\", \"_rank\": 5, \"_sweep\": \"alignment\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 2.356194490192345, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.0, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"7f759e72b65e\", \"mode\": \"alignment\", \"sweep\": \"alignment\", \"reps\": 800, \"raw_path\": \"data/sim/alignment/raw/7f759e72b65e.parquet\", \"means_path\": \"data/sim/alignment/means/7f759e72b65e.npz\", \"_rank\": 5, \"_sweep\": \"alignment\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 2.6179938779914944, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.0, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"b8d6eec61a85\", \"mode\": \"alignment\", \"sweep\": \"alignment\", \"reps\": 800, \"raw_path\": \"data/sim/alignment/raw/b8d6eec61a85.parquet\", \"means_path\": \"data/sim/alignment/means/b8d6eec61a85.npz\", \"_rank\": 5, \"_sweep\": \"alignment\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 3, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579], \"theta\": 2.8797932657906435, \"profile\": \"mixed\", \"label\": \"main\", \"g\": 1.0, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"b9beaff1ecc0\", \"mode\": \"alignment\", \"sweep\": \"alignment\", \"reps\": 800, \"raw_path\": \"data/sim/alignment/raw/b9beaff1ecc0.parquet\", \"means_path\": \"data/sim/alignment/means/b9beaff1ecc0.npz\", \"_rank\": 5, \"_sweep\": \"alignment\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 32, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(32), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)